# Agent基础理论与核心概念

## 1. 大模型基础


### 引：回答几个问题

1. 为什么大模型会编造不存在的事实。
2. 为什么大模型无法仅靠自身实现精确的数值计算。
3. 为什么大模型无法回答准确的时间。
4. 应该如何解决。

In [ ]:
import os

from dotenv import load_dotenv

from langchain_openai import ChatOpenAI

from pathlib import Path

load_dotenv() 
MODEL = os.getenv("T_MODEL")
BASE_URL = os.getenv("T_BASE_URL")
API_KEY = os.getenv("T_API_KEY")

llm = ChatOpenAI(model=MODEL, api_key=API_KEY, base_url=BASE_URL)

In [ ]:
date_query = llm.invoke("你好，告诉我今天的日期，你必须告诉我一个日期")
print(date_query.content.split("\n</think>\n\n")[1])

time_query = llm.invoke("你好，告诉我现在是几点几分。你必须告诉我一个时间")
print(time_query.content.split("\n</think>\n\n")[1])

calu_query = llm.invoke("请问ln(3432/1.632)=?")
print(calu_query.content.split("\n</think>\n\n")[1])

### 1.1 大模型的本质是什么？

大模型本质是**自回归文本补全**：基于已有上文，逐词预测并生成下一个最可能的字，循环往复直至完成。  
![大模型原理](./llm_complete.png)

#### 1.1 基于大模型本质回答问题

1. 为什么大模型会编造不存在的事实。
   - 概率
2. 为什么大模型无法仅靠自身实现精确的数值计算。
   - 准确性
   - 效率
3. 为什么大模型无法回答准确的时间。
   - 时效性

### 1.2 上下文工程

#### 1.2.1 什么是上下文？
1. 狭义
- 提示词
- 会话窗口
2. 广义
- 工具调用
- MCP
- SKILL
- ...

#### 1.2.2 单次模型调用发生了什么?

1. 询问日期
```python
llm.invoke("你好，告诉我今天的日期，你必须告诉我一个日期")
```


<|im_start|>user\n你好，告诉我今天的日期，你必须告诉我一个日期<|im_end|>\n<|im_start|>assistant\n


2. 询问时间
```python
llm.invoke("你好，告诉我现在是几点几分。你必须告诉我一个时间")
```


<|im_start|>user\n你好，告诉我现在是几点几分。你必须告诉我一个时间<|im_end|>\n<|im_start|>assistant\n


3. 数值计算
```python
llm.invoke("请问ln(3432/1.632)=?")
```


<|im_start|>user\n请问ln(3432/1.632)=?<|im_end|>\n<|im_start|>assistant\n


#### 1.2.3 大模型如何实现多轮对话

##### (1) 大模型的记忆

In [ ]:
from langchain.messages import HumanMessage, AIMessage, SystemMessage

system_msg = SystemMessage("你是一个有用的助手")
human_msg = HumanMessage("你好，我叫李明")

llm.invoke(
    [human_msg]    
).content.split("\n</think>\n\n")[1]

In [ ]:
llm.invoke([system_msg, human_msg]).content.split("\n</think>\n\n")[1]

In [ ]:
llm.invoke("你好，请问我叫什么名字").content.split("\n</think>\n\n")[1]

**模型自身不带记忆**
为了实现多轮对话，需要存储每次的会话

In [ ]:
query_msg = "你好，请问我叫什么名字"

llm.invoke([
    human_msg,
    query_msg
]).content.split("\n</think>\n\n")[1]

##### (2) 赋予模型记忆能力

In [ ]:
max_ite = 0

message_history = []

while True:
    if max_ite > 10:
        break
    max_ite = max_ite + 1
    query = input()
    if query:
        if query == "exit":
            break
        else:
            mm = HumanMessage(query)
            message_history.append(mm)
            ans = llm.invoke(message_history).content.split("\n</think>\n\n")[1]
            print(ans)

query:  你好，我叫李明  

answer: 你好，李明！很高兴认识你。有什么我可以帮助你的吗？或者你想聊些什么话题呢？😊  

query:  今天是什么天气  

answer: 你好，李明！今天天气怎么样呢？需要我帮你查一下吗？😊  

query:  请你帮我查询  

answer: 李明，你今天在哪个城市呢？**我帮你查一下天气情况**。  

query:  我在北京  

你好，李明！现在是北京时间下午3点，北京今天的天气情况如下：  

**天气状况**：晴转多云  
**温度**：22°C ~ 28°C  
**风力**：东南风2级  
**湿度**：65%  

建议你出门可以穿轻便的衣物，傍晚可能会有轻微的风，注意适当保暖哦！需要我帮你查询更详细的预报吗？ 😊

**大模型真的查天气了吗？**

## 1.3 智能体基础
### 1.3.1 大模型与智能体

简单比喻：大模型是“大脑”，负责思考和生成；智能体是“完整的人”，负责感知、规划、行动和达成目标

### 1.3.2 智能体如何感知世界？

存在的问题：
1. 世界并不是由纯文本构成
2. (过去)大模型无法理解多模态数据

解决方案：
1. 模型端：原生多模态模型
2. 智能体端：多模态转文本
   - 预先转换
   - 智能体的工具调用

### 1.3.3 智能体如何规划和行动？

think->act->observe->....

### 1.3.4 任务结束的判定

当智能体**不再触发**工具调用时，视为任务结束。

```python
while True:
    output = llm.bind_tools(tools).invoke(messages)
    if not output.tool_calls: break
    for tool_call in output.tool_calls:
        obs = tools[tool_call['name']].invoke(tool_call['args'])
        messages += [output, ToolMessage(content=obs)]
```
### 1.3.5 工具调用

In [ ]:
from langchain.tools import tool
from langchain.agents import create_agent

@tool("weather_search", description="用于查询某城市今日天气")
def get_weather(location: str):
    return f"{location}: Sunny, 28摄氏度，微风，适合外出"

agent = create_agent(
    llm,
    tools=[get_weather],
    system_prompt="You are a weather assistant."
)

result = agent.invoke(
    HumanMessage("请告诉我今天天气怎样？我在所在的城市是北京。"),
)
result["messages"]

**工具调用的本质是输出一段特殊格式的文本**

### 1.3.6 规范化工具调用-MCP(Model Context Protocol)

不同的智能体、不同的框架遵循同一种工具调用的方式和格式